In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")





db = client.get_database('telematics_db')








gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x)
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)



In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity='all'
%matplotlib inline

In [ ]:

def loc_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]

    try:
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
#        print(url)
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(vid)])
        #print("LOC PINGS A GAYI")
        
    except Exception as e:
#        print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df


In [ ]:
def hb_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:        
        url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        hb_data=json.load(urllib.request.urlopen(url))
        hb=pd.DataFrame(hb_data['data'][str(vid)])
        #print("HB AAYA")

    except Exception as e:

#        print("hb pings issue >> " + str(e))
        hb=pd.DataFrame()
        
    return hb



In [ ]:
def ping_analysis(l):
#if 1>0:
    try:
        vid=l[0]
        lc=loc_pings(l)
        hb=hb_pings(l)
        print(vid)

        if (len(hb)==0)&(len(lc)>0):
            lc['date_time']=lc['time'].apply(getdate)
            lc.set_index('date_time',inplace=True)
            T_counts_array=lc.location.resample('T').count().values
            no_data_count=len(T_counts_array[T_counts_array==0])+1440-len(T_counts_array)
            min_wise=(1440-no_data_count)/60
            lc['created']=lc['created']/1000
            lc.sort_values('created',inplace=True,ascending=True)
            lc['created_time_diff']=lc['created']-lc['created'].shift(1)
            live_time=lc[lc['created_time_diff']<5*60]['created_time_diff'].sum()/3600
            heart_beat='No heartbeat'

        elif (len(hb)>0):
            hb['created']=hb['created']/1000
            hb_times=hb['created']
            try:
                lc_times=lc['time']
                complete_times=hb_times.append(lc_times)

            except:
                complete_times=hb_times

            random=list(range(0,len(complete_times),1))
            combined=pd.DataFrame()
            combined['time']=complete_times
            combined['random']=random
            combined['date_time']=combined['time'].apply(getdate)
            combined.set_index('date_time',inplace=True)
            T_counts_array=combined.time.resample('T').count().values
            no_data_count=len(T_counts_array[T_counts_array==0])+1440-len(T_counts_array)
            min_wise=(1440-no_data_count)/60

            hb.sort_values('created',inplace=True,ascending=True)
            hb['created_time_diff']=hb['created']-hb['created'].shift(1)
            hb=hb[1:]
            live_time=hb[hb['created_time_diff']<15*60]['created_time_diff'].sum()/3600
            heart_beat='Has heartbeat'


        elif (len(hb)==0)&(len(lc)==0):
            min_wise=0
            live_time=0
            heart_beat='No data'


        return [vid,min_wise,live_time,heart_beat,getDay((l[2]+l[1])/2),pd.datetime.today().date()]
    except Exception as e:
        print("issue for vehicle:" + str(vid) +":  " + str(e))

In [ ]:

vehicle_details=pd.read_sql("select  vehicle_id,vehicle_number,model_name,operator_name from vehicle_details where vehicle_state='LIVE'",con=engine)
vehicle_list=list(vehicle_details['vehicle_id'])
start=int(gettime(pd.datetime.today().date()-timedelta(days=1)))
end=int(gettime(pd.datetime.today().date()))
vehicle_list=[[x,start,end] for x in vehicle_list]


In [ ]:
final=[]
for i in range(0,len(vehicle_list)-1000,1000):
    import multiprocessing
    pool = multiprocessing.Pool(100)
    results1=pool.map(ping_analysis,vehicle_list[i:(i+1000)])
    pool.close()
    pool.join()
    final+=results1
    print(data)
    break

data=pd.DataFrame(final,columns=['vehicle_id','1_min_data','15_min_live_tracking','data_status','day','analysis_done_on'])
data.to_csv("data_consistency_metric.csv")

In [ ]:
a=pd.read_csv("data_consistency_metric_1min_5_min.csv")

In [ ]:
a.data_status.value_counts()

In [ ]:
a['1_min_data']=a['1_min_data']*100

In [ ]:
plt.hist(a[a['1_min_data']>0]['1_min_data'],bins=range(0,100,1))
a[a['1_min_data']>0]['1_min_data'].describe(percentiles=[0.1,0.12,0.14,0.15,0.18,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95,0.99])

In [ ]:
a['5_min_live_tracking']=a['5_min_live_tracking']/864

In [ ]:
plt.hist(a[a['1_min_data']>0]['5_min_live_tracking'],bins=range(0,100,1))

a[a['1_min_data']>0]['5_min_live_tracking'].describe(percentiles=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95,0.99])

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
vehicle_details=pd.read_sql("select vehicle_id,model_name from vehicle_details where vehicle_state='LIVE'",con=engine)

In [ ]:
a=pd.merge(a,vehicle_details,on='vehicle_id')

In [ ]:
a_data=a[a['data_status']!='No data']

In [ ]:
plt.hist(a['15_min_live_tracking'],bins=24*4)
plt.rcParams["figure.figsize"] = [12,12]

In [ ]:
a_data['15_min_live_tracking'].describe(percentiles=[0.1,0.2,0.3,0.4,0.7,0.8,0.9,0.95,0.99])

In [ ]:
a_data[a_data['model_name']=='FMB920']['1_min_data'].describe(percentiles=[0.1,0.2,0.4,0.7,0.8,0.9,0.95,0.99])

In [ ]:
a_data[a_data['model_name']=='GT06']['15_min_live_tracking'].describe(percentiles=[0.1,0.2,0.4,0.7,0.8,0.9,0.95,0.99])  

In [ ]:
models_12less=a_data[a_data['1_min_data']<12]['model_name'].value_counts().keys()
values_12less=a_data[a_data['1_min_data']<12]['model_name'].value_counts().values

In [ ]:
vd_models=a_data['model_name'].value_counts().keys()
vd_values=a_data['model_name'].value_counts().values

In [ ]:
less12=pd.DataFrame()
less12['model_name']=models_12less
less12['12less_count']=values_12less
vd_count=pd.DataFrame()
vd_count['model_name']=vd_models
vd_count['count_total']=vd_values
final=pd.merge(less12,vd_count,on='model_name',how='left')

In [ ]:
final['percentage']=100*final['12less_count']/final.count_total

In [ ]:
final.sort_values('percentage',ascending=False)

In [ ]:
plt.hist(a_data['vehicle_id'],bins=range(0,120000,1000))
#fi
##fig_size[0] = 12
#fig_size[1] = 9
plt.rcParams["figure.figsize"] = [12,12]

In [ ]:
model_name='BW08'
plt.scatter(a_data[a_data['model_name']==model_name]['1_min_data'],a_data[a_data['model_name']==model_name]['vehicle_id'],alpha=0.3,s=a_data[a_data['model_name']==model_name]['15_min_live_tracking'].apply(lambda x: x*x),linewidths=0.1)   
plt.xlabel("data consistency from 0-24 hours")
plt.ylabel("range of vehicle_IDs")
plt.title("atleast 1 ping per minute for TL250")

In [ ]:
plt.scatter(a_data['1_min_data'],a_data['15_min_live_tracking'],alpha=0.1)   
plt.xlabel("data consistency from 0-24 hours")
plt.ylabel("live tracking from 0-24 hours")
plt.title("atleast 1 ping per minute metric")
plt.rcParams["figure.figsize"] = [12,12]

In [ ]:
model_name='BW08'
plt.scatter(a_data[a_data['model_name']==model_name]['1_min_data'],a_data[a_data['model_name']==model_name]['15_min_live_tracking'],alpha=0.2)   
plt.xlabel("data consistency from 0-24 hours")
plt.ylabel("live tracking from 0-24 hours")
plt.title("METRIC FOR {}".format(model_name))
plt.rcParams["figure.figsize"] = [12,12]
print("total number of vehicles for {}: ".format(model_name)+str(len(a_data[a_data['model_name']==model_name])))

In [ ]:
pd.read_sql("select * from vehicle_details limit 100",con=engine)

In [ ]:
pd.read_sql("select * from vehicle_details limit 100",con=engine)

In [ ]:
deboarded=pd.DataFrame(list(db.auditLog.find({'action':'OP_NO_CHANGE'})))

In [ ]:
vehicle_details=pd.read_sql("select operator_phone_number,operator_name,vehicle_id,vehicle_number,operator_id from vehicle_details where operator_id in {}".format(tuple(deboarded['entityId'])),con=engine)        

 

In [ ]:
final=pd.merge(deboarded,vehicle_details,how='left',left_on='entityId',right_on='operator_id')

In [ ]:
final.to_csv("nawal_op_no_change.csv")

In [ ]:
final.to_csv("samiya_deboard_dump.csv")

In [ ]:
pd.read_sql("select * from wire_issues limit 100",con=engine)